In [5]:
import os
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# ====================================================================
# 1. GENERALIZED DATA LOADER AND SUMMARIZER
# ====================================================================
def load_and_summarize_data(file_path):
    # Scan the first few lines to detect metadata headers
    skip_rows = 0
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        for idx, line in enumerate(f):
            if any(key in line for key in ['Scan Num', '101 (', 'Scan Swee', 'timestamp']):
                skip_rows = idx
                break
                
    # Load data skipping the identified metadata rows
    df = pd.read_csv(file_path, skiprows=skip_rows)
    df.columns = df.columns.str.strip()
    df = df.dropna(how='all', axis=1).dropna(how='all', axis=0)
    
    # Identify the time column dynamically
    time_cols = [col for col in df.columns if 'time' in col.lower() or 'swee' in col.lower()]
    df['Timestamp'] = df[time_cols[0]] if time_cols else df.index
    
    # Identify sensor channels dynamically (looking for Celsius or channel indicators)
    sensor_cols = [col for col in df.columns if 'C' in col or 'ch_' in col.lower()]
    
    # Scrub hardware overloads and impute missing gaps
    for col in sensor_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        overload_mask = (df[col].abs() > 10000) | (df[col] == np.inf)
        df.loc[overload_mask, col] = np.nan
        
    df[sensor_cols] = df[sensor_cols].fillna(method='ffill').fillna(method='bfill')
    
    # Calculate average time interval between data points
    try:
        numeric_timestamps = pd.to_numeric(df['Timestamp'], errors='coerce')
        avg_interval = numeric_timestamps.diff().mean()
        interval_display = f"{avg_interval:.2f} units"
    except Exception:
        interval_display = "Categorical or irregular timestamps"

    # Print Data Summary
    print("\n==================================================")
    print(" DATASET SUMMARY")
    print("==================================================")
    print(f"Total Data Points (Rows): {len(df)}")
    print(f"Total Sensor Channels   : {len(sensor_cols)}")
    print(f"Average Time Interval   : {interval_display}")
    print("==================================================\n")
        
    return df, sensor_cols

# ====================================================================
# 2. SEQUENCE GENERATION
# ====================================================================
def create_sequences(df_scaled, timestamps, time_steps):
    # Converts 2D tabular data into 3D tensors for LSTM processing
    Xs, ts = [], []
    for i in range(len(df_scaled) - time_steps):
        Xs.append(df_scaled.iloc[i:(i + time_steps)].values)
        ts.append(timestamps.iloc[i + time_steps - 1])
    return np.array(Xs), np.array(ts)

# ====================================================================
# 3. DYNAMIC LSTM ARCHITECTURE
# ====================================================================
class DynamicLSTMAutoencoder(nn.Module):
    def __init__(self, num_features):
        super(DynamicLSTMAutoencoder, self).__init__()
        
        # Dynamically scale the bottleneck based on input complexity
        # Minimum of 16, scales up to 64 maximum for high channel counts
        self.hidden_size = min(64, max(16, num_features * 4))
        
        self.encoder_lstm = nn.LSTM(input_size=num_features, hidden_size=self.hidden_size, batch_first=True)
        self.decoder_lstm = nn.LSTM(input_size=self.hidden_size, hidden_size=self.hidden_size, batch_first=True)
        self.output_layer = nn.Linear(self.hidden_size, num_features)
        
    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        
        # Compress sequence
        _, (hidden_state, _) = self.encoder_lstm(x)
        last_hidden_state = hidden_state[-1]
        
        # Prepare latent vector for decoding
        repeated_hidden = last_hidden_state.unsqueeze(1).repeat(1, seq_len, 1)
        
        # Reconstruct sequence
        decoded, _ = self.decoder_lstm(repeated_hidden)
        reconstructed = self.output_layer(decoded)
        
        return reconstructed

# ====================================================================
# 4. TRAINING ENGINE
# ====================================================================
def train_autoencoder(model, train_loader, num_epochs=30, learning_rate=0.001):
    print(f"Training LSTM Autoencoder with hidden dimension: {model.hidden_size}")
    criterion = nn.L1Loss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    
    model.train()
    for epoch in range(num_epochs):
        for batch_x in train_loader:
            optimizer.zero_grad()
            outputs = model(batch_x[0])
            loss = criterion(outputs, batch_x[0])
            loss.backward()
            optimizer.step()
            
    print("Model training completed successfully.")
    return model

# ====================================================================
# 5. ANOMALY DETECTION AND CONFIDENCE SCORING
# ====================================================================
def detect_anomalies(model, X_seq_tensor, df_timestamps, df_original, sensor_cols, contamination=0.03):
    model.eval()
    with torch.no_grad():
        reconstructed = model(X_seq_tensor)
        
    X_seq_np = X_seq_tensor.numpy()
    reconstructed_np = reconstructed.numpy()
    
    # Calculate Mean Absolute Error across the entire sequence window
    mae_loss = np.mean(np.abs(reconstructed_np - X_seq_np), axis=(1, 2))
    
    # Set mathematical threshold based on contamination factor
    threshold = np.percentile(mae_loss, (1 - contamination) * 100)
    max_mae = np.max(mae_loss)
    
    results_df = pd.DataFrame({
        'Timestamp': df_timestamps,
        'MAE_Score': mae_loss,
        'Is_Anomaly': mae_loss > threshold
    })
    
    # Calculate Confidence Score scaled from 50 percent to 100 percent for flagged anomalies
    # If the MAE perfectly equals the threshold, it is 50 percent confident. 
    # If the MAE equals the maximum error ever seen, it is 100 percent confident.
    def calculate_confidence(mae):
        if mae <= threshold:
            return 0.0
        if max_mae == threshold:
            return 100.0
        scaled = 50.0 + ((mae - threshold) / (max_mae - threshold)) * 50.0
        return min(100.0, scaled)
        
    results_df['Confidence_Score'] = results_df['MAE_Score'].apply(calculate_confidence)
    
    # Merge with original data to extract physical reading values at the anomaly timestamp
    merged_df = pd.merge(results_df, df_original, on='Timestamp', how='inner')
    anomalies_only = merged_df[merged_df['Is_Anomaly'] == True]
    
    print("\n==================================================")
    print(" DETECTED ANOMALIES REPORT")
    print("==================================================")
    for index, row in anomalies_only.iterrows():
        print(f"Timestamp: {row['Timestamp']} | Confidence: {row['Confidence_Score']:.1f} %")
        for col in sensor_cols:
            print(f"   * {col}: {row[col]:.2f}")
        print("==================================================")
        
    return merged_df, threshold

# ====================================================================
# 6. VISUALIZATION DASHBOARD
# ====================================================================
def plot_channel_anomalies(merged_df, sensor_cols, file_name):
    anomalies = merged_df[merged_df['Is_Anomaly'] == True]
    num_sensors = len(sensor_cols)
    
    fig = make_subplots(
        rows=num_sensors, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.04,
        subplot_titles=[f"Channel Overlay: {col}" for col in sensor_cols]
    )
    
    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22']
    
    for i, col in enumerate(sensor_cols):
        row_idx = i + 1
        
        # Plot continuous sensor data
        fig.add_trace(
            go.Scatter(
                x=merged_df['Timestamp'], y=merged_df[col],
                mode='lines', line=dict(color=colors[i % len(colors)], width=1.5),
                name=col, showlegend=False
            ), row=row_idx, col=1
        )
        
        # Overlay anomalies with interactive confidence score hover text
        hover_template = (
            "Timestamp: %{x}<br>" +
            f"Value: %{{y:.2f}}<br>" +
            "Confidence: %{customdata:.1f}%<extra></extra>"
        )
        
        fig.add_trace(
            go.Scatter(
                x=anomalies['Timestamp'], y=anomalies[col],
                mode='markers', customdata=anomalies['Confidence_Score'],
                marker=dict(color='red', size=8, symbol='diamond', line=dict(width=1, color='darkred')),
                hovertemplate=hover_template, name=f"{col} Anomaly", showlegend=False
            ), row=row_idx, col=1
        )
        
        fig.update_yaxes(title_text="Value", row=row_idx, col=1)

    fig.update_layout(
        title=f"Generalized Anomaly Detection Dashboard: {file_name}",
        height=300 + (200 * num_sensors),
        width=1250,
        hovermode='x unified'
    )
    fig.show()

# ====================================================================
# 7. MAIN EXECUTION PIPELINE
# ====================================================================
def run_pipeline(file_path):
    if not os.path.exists(file_path):
        print("Error: File path does not exist.")
        return
        
    file_name = os.path.basename(file_path)
    
    # Step 1: Load and summarize
    df, sensors = load_and_summarize_data(file_path)
    
    if len(sensors) == 0:
        print("Error: No valid sensor channels detected.")
        return

    # Step 2: Scale data
    scaler = StandardScaler()
    df_scaled = pd.DataFrame(scaler.fit_transform(df[sensors]), columns=sensors)
    
    # Step 3: Create time windows
    TIME_STEPS = 10
    X_seq, timestamps_seq = create_sequences(df_scaled, df['Timestamp'], TIME_STEPS)
    X_tensor = torch.tensor(X_seq, dtype=torch.float32)
    
    dataset = TensorDataset(X_tensor)
    train_loader = DataLoader(dataset, batch_size=16, shuffle=False)
    
    # Step 4: Build dynamic model and train
    model = DynamicLSTMAutoencoder(num_features=len(sensors))
    trained_model = train_autoencoder(model, train_loader, num_epochs=30)
    
    # Step 5: Detect anomalies and calculate confidence
    merged_results, threshold = detect_anomalies(trained_model, X_tensor, timestamps_seq, df, sensors)
    
    # Step 6: Render plots
    plot_channel_anomalies(merged_results, sensors, file_name)

# Example execution call
run_pipeline(r"C:\Users\hari7\Documents\Anamoly Detection\uploads\uploads\cataluminiscence\1781339239_2 2025-11-21 0.csv")

C:\Users\hari7\AppData\Local\Temp\ipykernel_3384\190239408.py:42: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df[sensor_cols] = df[sensor_cols].fillna(method='ffill').fillna(method='bfill')



 DATASET SUMMARY
Total Data Points (Rows): 2770
Total Sensor Channels   : 1
Average Time Interval   : nan units

Training LSTM Autoencoder with hidden dimension: 16
Model training completed successfully.

 DETECTED ANOMALIES REPORT
Timestamp: 2025-11-21 13:44:37.951 | Confidence: 52.9 %
   * 101 (°C): 23.43
Timestamp: 2025-11-21 13:44:38.951 | Confidence: 58.8 %
   * 101 (°C): 23.44
Timestamp: 2025-11-21 13:44:39.951 | Confidence: 58.9 %
   * 101 (°C): 23.45
Timestamp: 2025-11-21 13:44:40.951 | Confidence: 58.5 %
   * 101 (°C): 23.46
Timestamp: 2025-11-21 13:44:41.951 | Confidence: 58.3 %
   * 101 (°C): 23.47
Timestamp: 2025-11-21 13:44:42.951 | Confidence: 55.9 %
   * 101 (°C): 23.44
Timestamp: 2025-11-21 13:44:43.951 | Confidence: 57.1 %
   * 101 (°C): 23.47
Timestamp: 2025-11-21 13:44:44.951 | Confidence: 56.5 %
   * 101 (°C): 23.46
Timestamp: 2025-11-21 13:44:45.951 | Confidence: 56.5 %
   * 101 (°C): 23.44
Timestamp: 2025-11-21 13:44:46.951 | Confidence: 56.6 %
   * 101 (°C): 23.